# 03 — Silver: cleaned and typed listings per city

In [0]:
dbutils.widgets.text("city", "sicily")
city = dbutils.widgets.get("city").strip().lower()

In [0]:
import math
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze.listings"
SILVER = f"{CATALOG}.silver.listings"
QUARANTINE = f"{CATALOG}.silver.listings_quarantine"

MIN_PRICE = 10
MAX_PRICE = 5000
MAX_GUESTS = 16
LONG_STAY = 28
IQR_K = 1.5

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

In [0]:
NUMERIC = ["latitude", "longitude", "accommodates", "bedrooms", "beds",
           "minimum_nights", "maximum_nights",
           "availability_30", "availability_60", "availability_90", "availability_365",
           "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d", "reviews_per_month",
           "host_listings_count", "host_total_listings_count", "calculated_host_listings_count",
           "calculated_host_listings_count_entire_homes", "calculated_host_listings_count_private_rooms",
           "calculated_host_listings_count_shared_rooms",
           "estimated_occupancy_l365d", "estimated_revenue_l365d",
           "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
           "review_scores_checkin", "review_scores_communication", "review_scores_location",
           "review_scores_value", "hosts_time_as_host_years", "hosts_time_as_host_months"]
MONEY = ["price"]
PERCENT = ["host_acceptance_rate"]
BOOL = ["host_is_superhost", "host_has_profile_pic", "host_identity_verified", "has_availability"]
DATES = ["last_scraped", "first_review", "last_review"]
STRINGS = ["listing_url", "name", "room_type", "property_type",
           "neighbourhood_cleansed", "neighbourhood_group_cleansed",
           "bathrooms_text", "license", "amenities"]
REVIEW_SCORES = [c for c in NUMERIC if c.startswith("review_scores_")]

In [0]:
bronze = spark.table(BRONZE).filter(F.col("city") == city)
cols = set(bronze.columns)

def src(c, dtype="string"):
    return F.col(f"`{c}`") if c in cols else F.lit(None).cast(dtype)

def num(c):
    return F.expr(f"try_cast(`{c}` AS DOUBLE)") if c in cols else F.lit(None).cast("double")

def money(c):
    return F.expr(f"try_cast(regexp_replace(`{c}`, '[^0-9.]', '') AS DOUBLE)") if c in cols else F.lit(None).cast("double")

def percent(c):
    return F.expr(f"try_cast(regexp_replace(`{c}`, '%', '') AS DOUBLE) / 100") if c in cols else F.lit(None).cast("double")

def boolean(c):
    v = F.lower(F.trim(src(c)))
    return F.when(v.isin("t", "true"), True).when(v.isin("f", "false"), False)

def date(c):
    return F.expr(f"try_cast(left(`{c}`, 10) AS DATE)") if c in cols else F.lit(None).cast("date")

def string(c):
    return F.expr(f"nullif(trim(`{c}`), '')") if c in cols else F.lit(None).cast("string")

typed = bronze.select(
    "city", "snapshot_date",
    F.expr("try_cast(id AS BIGINT)").alias("id"),
    F.expr("try_cast(host_id AS BIGINT)").alias("host_id"),
    *[num(c).alias(c) for c in NUMERIC],
    *[money(c).alias(c) for c in MONEY],
    *[percent(c).alias(c) for c in PERCENT],
    *[boolean(c).alias(c) for c in BOOL],
    *[date(c).alias(c) for c in DATES],
    *[string(c).alias(c) for c in STRINGS],
)

In [0]:
for c in REVIEW_SCORES:
    typed = typed.withColumn(c, F.when(F.col(c).between(1, 5), F.col(c)))

silver = (typed
    .withColumn("bathrooms", F.when(F.lower("bathrooms_text").contains("half-bath"), F.lit(0.5))
                              .otherwise(F.expr("try_cast(regexp_extract(bathrooms_text, '([0-9]+[.]?[0-9]*)', 1) AS DOUBLE)")))
    .withColumn("bathrooms_shared", F.coalesce(F.lower("bathrooms_text").contains("shared"), F.lit(False)))
    .withColumn("host_tenure_months", F.col("hosts_time_as_host_years") * 12 + F.col("hosts_time_as_host_months"))
    .withColumn("has_price", F.col("price").isNotNull())
    .withColumn("no_reviews", F.coalesce(F.col("number_of_reviews"), F.lit(0)) == 0)
    .withColumn("min_nights_missing", F.col("minimum_nights").isNull())
    .withColumn("long_stay", F.coalesce(F.col("minimum_nights") >= LONG_STAY, F.lit(False)))
    .withColumn("license_status", F.when(F.col("license").isNull(), "missing")
                                   .when(F.lower("license").contains("exempt"), "exempt")
                                   .otherwise("licensed"))
    .withColumn("has_license", F.col("license_status") == "licensed")
    .withColumn("amenities_count", F.when(F.col("amenities").isNotNull(),
                                          F.size(F.from_json("amenities", "array<string>"))))
    .withColumn("log_price", F.when(F.col("price") > 0, F.log("price"))))

In [0]:
q1, q3 = silver.filter("log_price IS NOT NULL").approxQuantile("log_price", [0.25, 0.75], 0.001)
lo_log, hi_log = q1 - IQR_K * (q3 - q1), q3 + IQR_K * (q3 - q1)

silver = silver.withColumn("outlier_class", F
    .when(~F.col("has_price"), "no_price")
    .when((F.col("price") <= 0)
          | ~F.col("accommodates").between(1, MAX_GUESTS)
          | (F.col("minimum_nights") > F.col("maximum_nights"))
          | F.col("latitude").isNull() | F.col("longitude").isNull(), "error")
    .when((F.col("price") >= MAX_PRICE) | ((F.col("price") < MIN_PRICE) & ~F.col("long_stay")), "suspicious")
    .when((F.col("log_price") < lo_log) | (F.col("log_price") > hi_log), "tail")
    .otherwise("normal"))

In [0]:
received = silver.count()
deduped = silver.dropDuplicates(["id"])
duplicates_removed = received - deduped.count()

checked = deduped.withColumn("_reject_reason", F
    .when(F.col("id").isNull(), "bad_id")
    .when(F.col("host_id").isNull(), "bad_host_id"))

good = checked.filter(F.col("_reject_reason").isNull()).drop("_reject_reason")
bad = checked.filter(F.col("_reject_reason").isNotNull()).withColumn("_quarantined_at", F.current_timestamp())

if good.limit(1).count() == 0:
    raise ValueError(f"No valid rows for city '{city}'")

In [0]:
def write(df, table):
    w = df.write.format("delta").option("mergeSchema", "true")
    if spark.catalog.tableExists(table):
        w.mode("overwrite").option("replaceWhere", f"city = '{city}'").saveAsTable(table)
    else:
        w.mode("overwrite").partitionBy("city").saveAsTable(table)

write(good, SILVER)
write(bad, QUARANTINE)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

In [0]:
s = spark.table(SILVER).filter(F.col("city") == city)
delivered = s.count()
rejected = spark.table(QUARANTINE).filter(F.col("city") == city).count()
assert received == delivered + rejected + duplicates_removed, "rows received != delivered + rejected + duplicates"
print(f"{city}: received={received}, delivered={delivered}, rejected={rejected}, "
      f"duplicates_removed={duplicates_removed}, log-IQR fence=[{math.exp(lo_log):.0f}, {math.exp(hi_log):.0f}] EUR")
display(s.groupBy("outlier_class").agg(F.count("*").alias("rows"),
                                       F.percentile_approx("price", 0.5).alias("median_price"))
         .orderBy("outlier_class"))